# Introduction to Machine Learning – Lab 3 (Bonus)
## Interactive 3D Plots of the Clusters and the Price Classes

### Objective
- **Bonus a:** draw the houses on the first three principal components for every clustering (4 preparations × 4 variance thresholds), colored by cluster, with the centroids as a separate symbol. What do three dimensions show that two do not? Is k-means the right method for this data?
- **Bonus b:** the same plot, colored by the six price classes of Lab 2 (most expensive class in blue, the second most expensive in green, and so on). How do the price classes relate to the clusters, the principal components and the feature importance of the XGBoost model?

The figures are interactive: drag to rotate, scroll to zoom, click a legend entry to hide a group and use the dropdown at the top left to switch between the clusterings. Hovering over a house shows its price, quality and year of construction.

In [1]:
import sys
from pathlib import Path

# Add project root and notebooks/ to system path
sys.path.append(str(Path(".").resolve()))
sys.path.append(str(Path("notebooks").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from clustering import CLUSTER_COLORS, SEED, THRESHOLDS, fit_models, reduce_to_thresholds
from preprocessing import (
    CLASS_NAMES,
    ORDINAL,
    TARGET,
    drop_incomplete,
    fill_missing,
    load_raw,
    log_features,
    make_preprocessor,
    make_scaler,
    make_zscore_scaler,
    prepare_data,
)
from scipy.stats import spearmanr
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

# Interactive output for Jupyter and VS Code
pio.renderers.default = "plotly_mimetype"

DATA_PATH = Path("../data/AmesHousing.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/AmesHousing.csv")

N_CLUSTERS = 5
THRESHOLD = 0.90

# Price classes from the most expensive (blue) to the cheapest, as asked in the task
PRICE_COLORS = ["#CC79A7", "#D55E00", "#E69F00", "#F0E442", "#009E73", "#0072B2"]

### Data preparation
The four preparations of notebook 09 (min-max, z-score, each with and without the logarithm of the four skewed areas), each reduced once with PCA and clustered into 5 clusters at all four thresholds. The cluster numbers are the same as in notebooks 08 to 10.

In [2]:
X, y = prepare_data(DATA_PATH, drop_extreme=True, drop_near_constant=True)
X_complete, y_complete = drop_incomplete(X, y)
X_complete = fill_missing(X_complete)
price = load_raw(DATA_PATH).loc[X_complete.index, TARGET]
encoded_complete = make_preprocessor("onehot", min_frequency=5).fit_transform(X_complete, y_complete)

columns = encoded_complete.columns
onehot = [col for col in columns if col not in X_complete.columns]
ordinal = [col for col in columns if col in ORDINAL]
numeric = [col for col in columns if col not in onehot + ordinal]
scaled_columns = ordinal + numeric
logged = log_features(encoded_complete)

# The same four preparations as in notebook 09
prepared = {
    "Min-max": make_scaler().fit_transform(encoded_complete),
    "Z-score": make_zscore_scaler(scaled_columns).fit_transform(encoded_complete)[columns],
    "Min-max + log": make_scaler().fit_transform(logged),
    "Z-score + log": make_zscore_scaler(scaled_columns).fit_transform(logged)[columns],
}
fits = {name: reduce_to_thresholds(data) for name, data in prepared.items()}
models = {name: fit_models(reduced, counts, N_CLUSTERS) for name, (_, reduced, counts) in fits.items()}

rows = []
for name, (pca, reduced, counts) in fits.items():
    shares = pca.explained_variance_ratio_
    rows.append(
        {
            "Preparation": name,
            "PC1": shares[0], "PC2": shares[1], "PC3": shares[2],
            "Shown in 2D (of 90% kept)": shares[:2].sum() / shares[: counts[THRESHOLD]].sum(),
            "Shown in 3D (of 90% kept)": shares[:3].sum() / shares[: counts[THRESHOLD]].sum(),
        }
    )
display(pd.DataFrame(rows).set_index("Preparation").map("{:.1%}".format))

,PC1,PC2,PC3,Shown in 2D (of 90% kept),Shown in 3D (of 90% kept)
Preparation,,,,,
Min-max,13.4%,8.9%,5.5%,24.7%,30.8%
Z-score,18.3%,6.9%,5.2%,28.0%,33.7%
Min-max + log,13.5%,8.8%,5.4%,24.8%,30.8%
Z-score + log,18.7%,6.9%,5.4%,28.3%,34.3%


### Bonus a: clusters on PC1, PC2 and PC3
Each house is a point at its coordinates on PC1, PC2 and PC3. The black diamonds are the centroids, reduced to their first three coordinates. The variance threshold does not move the points (the first three components are the same at every threshold), it only changes the colors and the centroids slightly.

In [3]:
# Single precision keeps the stored figures small
hover = np.column_stack([price, X_complete["Overall Qual"], X_complete["Year Built"]]).astype(np.float32)
HOVER = "Price: $%{customdata[0]:,.0f}<br>Overall Qual: %{customdata[1]}<br>Year Built: %{customdata[2]}"


def point_traces(reduced, groups, names, colors, legend_title):
    """One 3D marker trace per group, so that every group can be switched off in the legend."""
    traces = []
    for number, (name, color) in enumerate(zip(names, colors)):
        mask = groups == number
        traces.append(
            go.Scatter3d(
                x=reduced[mask, 0].astype(np.float32), y=reduced[mask, 1].astype(np.float32), z=reduced[mask, 2].astype(np.float32),
                mode="markers", name=name, legendgroup=name, legendgrouptitle_text=legend_title if number == 0 else None,
                marker={"size": 2.6, "color": color, "opacity": 0.65},
                customdata=hover[mask], hovertemplate=f"{name}<br>{HOVER}<extra></extra>",
            )
        )
    return traces


def centroid_trace(model):
    """The centroids, reduced to their first three coordinates, as large black diamonds."""
    centers = model.cluster_centers_[:, :3]
    return go.Scatter3d(
        x=centers[:, 0], y=centers[:, 1], z=centers[:, 2],
        mode="markers+text", name="Centroids", text=[f"C{number}" for number in range(1, len(centers) + 1)],
        textposition="top center", textfont={"size": 13, "color": "black"},
        marker={"size": 9, "symbol": "diamond", "color": "black", "line": {"color": "white", "width": 2}},
        hovertemplate="Centroid %{text}<extra></extra>",
    )


def axis_titles(pca):
    shares = pca.explained_variance_ratio_
    return {f"scene.{axis}.title.text": f"PC{number} ({shares[number - 1]:.1%})" for number, axis in
            enumerate(["xaxis", "yaxis", "zaxis"], start=1)}


def dropdown_figure(views, title):
    """One figure with a dropdown: each view is (label, traces, layout changes); only one view is visible."""
    fig = go.Figure()
    spans = []
    for _, traces, _ in views:
        start = len(fig.data)
        for trace in traces:
            fig.add_trace(trace)
        spans.append((start, len(fig.data)))
    buttons = []
    for (label, _, layout), (start, end) in zip(views, spans):
        visible = [start <= index < end for index in range(len(fig.data))]
        buttons.append({"label": label, "method": "update", "args": [{"visible": visible}, {"title.text": f"{title}: {label}", **layout}]})
    first = buttons[0]["args"]
    for trace, visible in zip(fig.data, first[0]["visible"]):
        trace.visible = visible
    fig.update_layout(
        title={"text": first[1]["title.text"], "x": 0.5, "xanchor": "center", "y": 0.98},
        # The dropdown sits below the title, at the top left of the plot
        updatemenus=[{"buttons": buttons, "x": 0, "xanchor": "left", "y": 1.0, "yanchor": "top"}],
        height=760, margin={"l": 0, "r": 0, "t": 60, "b": 0},
        legend={"itemsizing": "constant", "groupclick": "toggleitem"},
        scene={"aspectmode": "cube"},
    )
    fig.update_layout(**{key: value for key, value in first[1].items() if key != "title.text"})
    return fig


views = []
for name, (pca, reduced, counts) in fits.items():
    for threshold in THRESHOLDS:
        model = models[name][threshold]
        traces = point_traces(reduced, model.labels_, [f"Cluster {n}" for n in range(1, N_CLUSTERS + 1)], CLUSTER_COLORS, "k-means")
        traces.append(centroid_trace(model))
        views.append((f"{name}, {threshold:.0%} ({counts[threshold]} components)", traces, axis_titles(pca)))

cluster_figure = dropdown_figure(views, "k-means clusters on PC1–PC3")
cluster_figure.show()

In [4]:
# Centroids on the first three components, and how far apart the clusters are with and without PC3
rows, pairs = [], []
for name in ["Min-max", "Z-score"]:
    pca, reduced, counts = fits[name]
    model = models[name][THRESHOLD]
    centers = model.cluster_centers_[:, :3]
    spread = reduced[:, :3].std(axis=0)
    for number, center in enumerate(centers, start=1):
        rows.append({"Preparation": name, "Cluster": number, **{f"PC{i + 1} (in std)": center[i] / spread[i] for i in range(3)}})
    for a in range(N_CLUSTERS):
        for b in range(a + 1, N_CLUSTERS):
            flat = np.linalg.norm((centers[a, :2] - centers[b, :2]) / spread[:2])
            deep = np.linalg.norm((centers[a] - centers[b]) / spread)
            pairs.append({"Preparation": name, "Pair": f"C{a + 1}–C{b + 1}", "Distance on PC1–PC2": flat,
                          "Distance on PC1–PC3": deep, "Gain from PC3": deep - flat})

print("Centroids, in standard deviations of the houses along each component:")
display(pd.DataFrame(rows).set_index(["Preparation", "Cluster"]).round(2))
pairs = pd.DataFrame(pairs)
print("Pairs of clusters that PC3 separates the most:")
display(pairs.sort_values("Gain from PC3", ascending=False).groupby("Preparation").head(3).set_index(["Preparation", "Pair"]).round(2))

for name in ["Min-max", "Z-score"]:
    pca = fits[name][0]
    weights = pd.Series(pca.components_[2] * np.sqrt(pca.explained_variance_[2]), index=columns)
    top = weights.abs().nlargest(6).index
    print(f"{name}, largest loadings on PC3: " + ", ".join(f"{col} ({weights[col]:+.2f})" for col in top))

Centroids, in standard deviations of the houses along each component:


PC1 (in std)  PC2 (in std)  PC3 (in std)
Preparation Cluster                                          
Min-max     1               -1.20         -0.91          0.50
            2                1.17         -1.04         -0.64
            3               -0.44          1.24         -0.02
            4                1.09          0.45          1.10
            5               -0.37         -0.01         -1.02
Z-score     1               -0.93          0.52          0.10
            2                1.11         -0.56         -0.23
            3               -1.64          0.70         -0.10
            4               -0.34         -0.84          0.35
            5                0.80          0.92         -0.33

Pairs of clusters that PC3 separates the most:


Distance on PC1–PC2  Distance on PC1–PC3  Gain from PC3
Preparation Pair                                                          
Min-max     C4–C5                 1.52                 2.61           1.09
            C2–C4                 1.49                 2.29           0.80
            C1–C5                 1.23                 1.95           0.73
Z-score     C2–C4                 1.47                 1.59           0.11
            C4–C5                 2.09                 2.20           0.11
            C1–C5                 1.77                 1.83           0.05

Min-max, largest loadings on PC3: House Style_1Story (+0.26), House Style_2Story (-0.23), Mas Vnr Type_BrkFace (-0.20), MS SubClass_60 (-0.19), Foundation_CBlock (-0.18), MS SubClass_20 (+0.17)
Z-score, largest loadings on PC3: Lot Area (+0.50), Fireplaces (+0.47), 1st Flr SF (+0.46), Lot Frontage (+0.45), Garage Yr Blt (-0.40), Year Built (-0.38)


#### Interpretation of the 3D plots
- **How much more the third axis shows:** PC3 adds 5.5% of the variance under min-max and 5.2% under z-score. The 3D plot shows 30.8% of the variance kept at 90% under min-max (24.7% in 2D) and 33.7% under z-score (28.0% in 2D). Most of the differences between the houses are still invisible, so clusters that touch in the plot can be apart in the other components.
- **Min-max: cluster 5 gets its own direction.** In the 2D plot of notebook 08, cluster 5 (the mixed houses of the 1970s) sat in the middle of the cloud, at −0.37 and −0.01 standard deviations on PC1 and PC2, and had the weakest silhouette. On PC3 its centroid lies at −1.02, the lowest of all clusters. In 3D it is no longer in the middle but below the others. PC3 contrasts one-story houses and vinyl siding with two-story houses, brick veneer, hardboard siding and cinder block, which is the mix that characterizes cluster 5.
- **Min-max: the two modern clusters separate.** Clusters 2 (modern two-story) and 4 (modern one-story) lie close together on PC1–PC2 (distance 1.49 standard deviations) and move apart on PC3 (cluster 4 at +1.10, cluster 2 at −0.64, distance 2.29). The pair 4–5 gains the most: from 1.52 to 2.61.
- **Z-score: PC3 adds little.** The z-score clusters are already separated on PC1 and PC2. The largest gain from PC3 is 0.11 standard deviations. PC3 (large old lots against new houses on small lots) runs across all clusters.
- **No gaps:** in every clustering the colors touch each other, and the cloud is one connected body with denser and thinner regions, not separate islands. The clusters are borders drawn through this body. This matches the silhouette values around 0.1 in notebooks 08 to 08.3.
- **Thresholds and logarithm:** switching between 98%, 95%, 90% and 80% changes the colors of only a few houses near the borders, and the log variants look almost the same as their counterparts, as notebook 09 measured (agreement 0.93 to 0.99).

#### Is k-means suitable for this data?
k-means assumes compact, roughly round groups of similar size, separated by empty space, and it puts every house into exactly one of them. Two other methods test these assumptions at 90% of the variance:
- **Gaussian mixture model (GMM):** each cluster is an ellipse of its own size and orientation, and a house belongs to every cluster with a probability. The number of clusters is chosen by the BIC (lower is better).
- **HDBSCAN:** a cluster is a region of high density, surrounded by regions of low density. Houses in no dense region are labeled noise. If the houses formed separate groups, HDBSCAN would find them.

In [5]:
rows = []
for name in ["Min-max", "Z-score"]:
    _, reduced, counts = fits[name]
    data = reduced[:, : counts[THRESHOLD]]
    kmeans = models[name][THRESHOLD].labels_

    # Gaussian mixture: elliptical clusters of any size and orientation, chosen by BIC
    bic = {}
    for k in range(1, 9):
        gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=SEED, n_init=2).fit(data)
        bic[k] = gmm.bic(data)
    best_k = min(bic, key=bic.get)
    gmm_labels = GaussianMixture(n_components=N_CLUSTERS, covariance_type="full", random_state=SEED, n_init=2).fit_predict(data)

    # HDBSCAN: clusters are dense regions separated by sparse ones, the rest is noise
    density = HDBSCAN(min_cluster_size=30, copy=True).fit_predict(data)
    found = len(set(density) - {-1})
    largest = np.bincount(density[density >= 0]).max() / len(density) if found else 0

    rows.append(
        {
            "Preparation": name,
            "k-means silhouette (5)": round(silhouette_score(data, kmeans), 3),
            "GMM: best k by BIC (1–8)": best_k,
            "GMM (5) vs k-means: ARI": round(adjusted_rand_score(kmeans, gmm_labels), 3),
            "GMM (5) silhouette": round(silhouette_score(data, gmm_labels), 3),
            "HDBSCAN: clusters": found,
            "HDBSCAN: noise": f"{(density == -1).mean():.0%}",
            "HDBSCAN: largest cluster": f"{largest:.0%}",
        }
    )
    print(f"{name}: GMM BIC by k: " + ", ".join(f"{k}: {value:,.0f}" for k, value in bic.items()))

display(pd.DataFrame(rows).set_index("Preparation").T)

Min-max: GMM BIC by k: 1: 100,591, 2: -13,396, 3: -55,091, 4: -60,151, 5: -86,589, 6: -118,853, 7: -88,972, 8: -93,436


Z-score: GMM BIC by k: 1: 360,535, 2: 265,084, 3: 236,457, 4: 225,715, 5: 235,280, 6: 213,334, 7: 218,995, 8: 216,304


Preparation,Min-max,Z-score
k-means silhouette (5),0.11,0.097
GMM: best k by BIC (1–8),6,6
GMM (5) vs k-means: ARI,0.772,0.371
GMM (5) silhouette,0.097,0.036
HDBSCAN: clusters,3,2
HDBSCAN: noise,86%,81%
HDBSCAN: largest cluster,8%,17%


#### Interpretation: is k-means the right method?
- **There are no separate groups to find.** HDBSCAN labels 86% (min-max) and 81% (z-score) of the houses as noise and finds only 3 and 2 small dense groups (the largest holds 8% and 17% of the houses). The houses form one continuous cloud, without empty space between natural groups. No clustering method can find well-separated clusters here, because there are none.
- **k-means therefore divides a continuum.** Its five clusters are a sensible **segmentation** of the houses into types (old houses, 1960s ranch houses, modern one-story houses, ...), as notebook 10 showed, and they are stable across seeds and thresholds. But the borders are drawn by the method and not by gaps in the data, which is why houses near a border, like the remodeled house E in notebook 10, can fall on either side.
- **The assumptions of k-means are only partly met.** The clusters are not separated by empty space but border each other directly, and the data mixes 0/1 columns with continuous ones, for which the Euclidean distance is a rough measure.
- **Gaussian mixture as an alternative:** the BIC prefers 6 components under both preparations. With 5 components the GMM agrees fairly well with k-means under min-max (ARI 0.772) but not under z-score (ARI 0.371), and its silhouette is lower (0.097 and 0.036 against 0.110 and 0.097), because its ellipses may overlap. Its advantage is the **probability** of each cluster: a house between two groups, like E, would get a sizable probability for both clusters instead of a hard label.
- **Conclusion:** k-means is acceptable to sort the houses into a few interpretable types, but not to discover natural groups, because the data has none. A Gaussian mixture with soft assignments, or a method made for mixed data (k-prototypes, or clustering with the Gower distance), would fit the data better. A density method like HDBSCAN is not useful here.

### Bonus b: the same plot, colored by price class
The points are the same as above, but colored by the six price classes of Lab 2: ≥ 400k blue, 250k–400k green, 180k–250k yellow, 130k–180k light orange, 100k–130k dark orange and < 100k pink. The black diamonds are the centroids of the 90% clustering of the selected preparation. The price was not used for the PCA or the clustering.

In [6]:
price_names = list(reversed(CLASS_NAMES))
price_colors = list(reversed(PRICE_COLORS))
# Classes are numbered from the most expensive, so the legend starts with the blue class
price_groups = (len(CLASS_NAMES) - 1 - y_complete).to_numpy()

views = []
for name, (pca, reduced, counts) in fits.items():
    traces = point_traces(reduced, price_groups, price_names, price_colors, "Price class")
    traces.append(centroid_trace(models[name][THRESHOLD]))
    views.append((f"{name} (centroids of the 90% clustering)", traces, axis_titles(pca)))

price_figure = dropdown_figure(views, "Price classes on PC1–PC3")
price_figure.show()

In [7]:
rows = []
for name, (pca, reduced, counts) in fits.items():
    labels = models[name][THRESHOLD].labels_
    six = KMeans(n_clusters=6, n_init=10, random_state=SEED).fit_predict(reduced[:, : counts[THRESHOLD]])
    rows.append(
        {
            "Preparation": name,
            **{f"Spearman PC{i + 1} / price": spearmanr(reduced[:, i], price).statistic for i in range(3)},
            "ARI 5 clusters / price classes": adjusted_rand_score(y_complete, labels),
            "AMI 5 clusters / price classes": adjusted_mutual_info_score(y_complete, labels),
            "ARI 6 clusters / price classes": adjusted_rand_score(y_complete, six),
        }
    )
display(pd.DataFrame(rows).set_index("Preparation").round(3))

for name in ["Min-max", "Z-score"]:
    table = pd.crosstab(
        pd.Series(models[name][THRESHOLD].labels_ + 1, name=f"{name} cluster"),
        pd.Series(pd.Categorical.from_codes(y_complete, CLASS_NAMES), name="Price class"),
    )
    print(f"{name}: houses per cluster and price class (90%, 5 clusters)")
    display(table)

,Spearman PC1 / price,Spearman PC2 / price,Spearman PC3 / price,ARI 5 clusters / price classes,AMI 5 clusters / price classes,ARI 6 clusters / price classes
Preparation,,,,,,
Min-max,0.796,0.015,-0.130,0.111,0.193,0.103
Z-score,0.922,0.096,0.066,0.191,0.283,0.197
Min-max + log,0.801,0.022,-0.129,0.112,0.192,0.115
Z-score + log,0.924,0.095,0.065,0.188,0.277,0.165


Min-max: houses per cluster and price class (90%, 5 clusters)


Price class,< 100k,100k–130k,130k–180k,180k–250k,250k–400k,≥ 400k
Min-max cluster,,,,,,
1,157,216,180,41,5,2
2,0,0,106,236,159,30
3,32,199,345,84,27,0
4,1,3,110,235,176,30
5,46,91,294,88,30,0


Z-score: houses per cluster and price class (90%, 5 clusters)


Price class,< 100k,100k–130k,130k–180k,180k–250k,250k–400k,≥ 400k
Z-score cluster,,,,,,
1,129,255,232,39,2,0
2,0,0,70,273,208,37
3,84,46,33,2,1,0
4,23,207,543,81,6,0
5,0,1,157,289,180,25


In [8]:
# The five features with the largest total gain in the XGBoost model of Lab 2 (notebook 03)
important = ["Overall Qual", "Gr Liv Area", "Year Built", "Total Bsmt SF", "Garage Area"]

rows = []
for name in ["Min-max", "Z-score"]:
    pca = fits[name][0]
    table = pd.DataFrame(pca.components_[:3].T * np.sqrt(pca.explained_variance_[:3]), index=columns, columns=["PC1", "PC2", "PC3"])
    squared = table**2
    for pc in table.columns:
        rows.append(
            {
                "Preparation": name,
                "Component": pc,
                **{col: round(table.loc[col, pc], 3) for col in important},
                "Share of the component from these 5": f"{squared.loc[important, pc].sum() / squared[pc].sum():.1%}",
                "Rank of Overall Qual": int(table[pc].abs().rank(ascending=False)["Overall Qual"]),
            }
        )
with pd.option_context("display.max_columns", None):
    display(pd.DataFrame(rows).set_index(["Preparation", "Component"]))

Overall Qual  Gr Liv Area  Year Built  Total Bsmt SF  \
Preparation Component                                                         
Min-max     PC1               0.112        0.064       0.187          0.064   
            PC2              -0.009       -0.038       0.029          0.052   
            PC3              -0.005       -0.043      -0.005          0.021   
Z-score     PC1               0.832        0.637       0.771          0.666   
            PC2               0.088        0.583      -0.148         -0.223   
            PC3              -0.079        0.366      -0.378          0.265   

                       Garage Area Share of the component from these 5  \
Preparation Component                                                    
Min-max     PC1              0.081                                4.5%   
            PC2              0.011                                0.6%   
            PC3             -0.004                                0.4%   
Z-score     PC1              0.741                               23.5%   
            PC2              0.002                                9.7%   
            PC3              0.094                               11.1%   

                       Rank of Overall Qual  
Preparation Component                        
Min-max     PC1                          28  
            PC2                         108  
            PC3                         122  
Z-score     PC1                           1  
            PC2                          41  
            PC3                          49

#### Interpretation: price classes against clusters
- **The price changes along PC1 and almost only along PC1.** The colors form layers across PC1: pink and orange on one side, green and blue on the other. The Spearman correlation of PC1 with the price is 0.796 under min-max and 0.922 under z-score. PC2 and PC3 hardly carry price information (|correlation| at most 0.13): the number of floors (PC2) and the lot size or materials (PC3) say little about the price.
- **The clusters are not the price classes.** The agreement between the 5 clusters and the 6 price classes is low: ARI 0.111 (min-max) and 0.191 (z-score), adjusted mutual information 0.193 and 0.283. Even with 6 clusters the ARI stays at 0.103 and 0.197. Every cluster spans four or five price classes, for example min-max cluster 3 (1960s ranch houses) has houses from < 100k to 250k–400k.
- **Why they differ:** the price classes cut the cloud only along PC1, into six slices. k-means cuts along PC1 and PC2 at the same time, because it uses all the variance. It therefore spends part of its five clusters on the number of floors, which does not change the price. The clearest example: the most expensive houses (≥ 400k) are split exactly in half between min-max clusters 2 and 4 (30 houses each), the modern two-story and the modern one-story houses. These two clusters have almost the same price mix but are different types of house. Along PC1 itself, the clusters only separate old, middle and new, which is coarser than six price classes.
- **Why z-score agrees better with the price:** its PC1 is built from exactly the features that decided the price in the XGBoost model of Lab 2. The five features with the largest total gain (notebook 03) are `Overall Qual`, `Gr Liv Area`, `Year Built`, `Total Bsmt SF` and `Garage Area`. Under z-score their loadings on PC1 are 0.83, 0.64, 0.77, 0.67 and 0.74, these 5 of 205 columns make up 23.5% of PC1, and `Overall Qual` has the largest loading of all. Under min-max the same loadings are only between 0.06 and 0.19, they make up 4.5% of PC1, and `Overall Qual` is only in place 28. The min-max PC1 is built from building materials, which follow the price only indirectly, through the age of the house.
- **Supervised against unsupervised:** XGBoost chose its features by how much they reduce the error of the price prediction. PCA and k-means do not know the price: they follow the directions with the most variance. A direction with a lot of variance, such as one story against two, need not have anything to do with the price. The clusters describe what kind of house it is; the price classes describe what it is worth. They overlap where type and worth go together (old houses are cheap, new ones expensive) and differ where they do not.
- **The no-garage cluster:** the z-score cluster 3 (houses without a garage) is the most price-homogeneous group: 130 of its 166 houses cost less than 130k, and 84 less than 100k. The missing garage, which XGBoost also used through `Garage Area` and `Garage Cars`, marks a cheap type of house.